[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/en/lab6/lab6_part2.ipynb)
# Lab 6: Convolutional neural networks — Making the CNN more complex


### Prerequisites. Install packages

We will use the same libraries as for Part 1. This time we will use the GPU, so we declare the `device` variable. Remember to send your models to the device when you instantiate them (`m = MyModel().to(device)`) and also to send the data tensors in the training loop (`X_batch = X_batch.to(device)` and the same for the labels).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
from torchvision import transforms
import numpy as np
import random
import os
from matplotlib import pyplot as plt

# Seed for reproducibility
seed = 1234567
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Loading the dataset

This time we will work with the *cifar10* image dataset, which consists of images with three channels.

In [ ]:
# CIFAR data are standardized using their mean and std, precomputed here.
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([
    transforms.ToTensor(),
    # TODO - Use transforms.Normalize to standardize
])

train_val = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# TODO - Split train_val into train and val (80/20)

# TODO - Create the DataLoaders
batch_size = 128
train_loader = ...
val_loader = ...
test_loader = ...

# Show one example from the set
images, labels = next(iter(train_loader))
print(images.shape)  # [128, 3, 32, 32]
print(labels.shape)  # [128]

# To display with pyplot, undo normalization and put channels last
def unnormalize(img):
    img = img.permute(1,2,0)  # CHW -> HWC
    img = img * torch.tensor(cifar10_std) + torch.tensor(cifar10_mean)
    return img.clamp(0,1)

plt.imshow(unnormalize(images[0]))
plt.xlabel(labels[0].item())
plt.show()


## Creating and training the model

For this problem we are going to use the following architecture:
1. [2D Convolution](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) of 32 filters and kernel size 3, with ReLU activation
1. [2D Pooling](https://docs.pytorch.org/docs/2.8/generated/torch.nn.MaxPool2d.html) taking the maximum of each group of 2x2
1. [2D Convolution](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) of 64 filters and kernel size 3, with ReLU activation
1. [2D Pooling](https://docs.pytorch.org/docs/2.8/generated/torch.nn.MaxPool2d.html) taking the maximum of each group of 2x2
1. [2D Convolution](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) of 64 filters and kernel size 3, with ReLU activation
1. Dense layer (requires prior flattening) of 64 units and ReLU activation
1. Output layer

Define the model and do the training as in the previous part.

In [ ]:
# TODO Define the model
# TODO Define the training and evaluation functions
# TODO Train
# TODO Inspect the training curves and measure test performance

If everything went well, you should have obtained a precision on test of at least 60%, which is not negligible for a simple network.

## Improving the performance

Go back to the architecture of the model and include [BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html) layers after each convolutional layer. This will make the outputs of that layer be standardized so that they follow a N(0,1) distribution (the standardization operation is included in the graph and, therefore, the computation of the gradients). The effect of this layer is that the gradient of the batch will not have a large component solely dedicated to bringing the outputs closer to the mean/standard deviation of the samples in each layer. Consequently, learning is accelerated.

### Exercises
 - Repeat the training with the new architecture. What effect have you noticed? Can you improve the test performance using concepts seen in previous labs?
 - Try different architectures and try to improve the test performance.

In [ ]:
# TODO Define the new model
# TODO Define the training and evaluation functions
# TODO Train
# TODO Inspect the training curves and measure test performance